# Phase 7 — Plusieurs témoins, un seul événement

## Objectifs

- Identifier les signalements correspondant potentiellement au même événement.
- Vérifier le problème créé par une découpe aléatoire.
- Réaliser une découpe train/test qui garde tous les relevés d'un même
  événement du même côté.
- Compter les témoignages textuellement identiques.
- Comparer les métriques avant et après la découpe par événements.

## Définition d'un événement

Deux relevés sont considérés comme parlant du même événement s'ils ont la même
date et heure d'observation, la même ville, le même État/région et le même pays.

## I. Imports

In [1]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline

## II. Chemins et constantes

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
PHASE7_DIR = OUTPUT_DIR / "phase_7_evenements"
PHASE7_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

RANDOM_STATE = 42
TEST_SIZE = 0.20

## III. Chargement robuste

In [3]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes chargées : {len(df)}")
print(f"Lignes mises à part : {len(lignes_problemes)}")

Lignes chargées : 88679
Lignes mises à part : 196


## IV. Conversions et cible

In [4]:
for col in ["duration_seconds", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["datetime", "date_posted"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

pattern_canular = "|".join(
    re.escape(mot) for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

## V. Variables du modèle sans fuite

In [5]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["text_features_without_leakage"] = (
    "city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

## VI. Créer l’identifiant d’événement

In [6]:
COLONNES_EVENEMENT = [
    "datetime",
    "city",
    "state",
    "country",
]

df["event_id"] = (
    df[COLONNES_EVENEMENT]
    .fillna("<MANQUANT>")
    .astype(str)
    .agg(" | ".join, axis=1)
)

df[
    COLONNES_EVENEMENT + ["event_id"]
].head()

,datetime,city,state,country,event_id
0,1949-10-10 20:30:00,san marcos,tx,us,1949-10-10 20:30:00 | san marcos | tx | us
1,1949-10-10 21:00:00,lackland afb,tx,,1949-10-10 21:00:00 | lackland afb | tx |
2,1955-10-10 17:00:00,chester (uk/england),,gb,1955-10-10 17:00:00 | chester (uk/england) | ...
3,1956-10-10 21:00:00,edna,tx,us,1956-10-10 21:00:00 | edna | tx | us
4,1960-10-10 20:00:00,kaneohe,hi,us,1960-10-10 20:00:00 | kaneohe | hi | us


## VII. Statistiques sue les évènements

In [7]:
taille_evenements = df["event_id"].value_counts()

nombre_evenements_plusieurs_temoins = int(
    (taille_evenements > 1).sum()
)

plus_grand_evenement = int(
    taille_evenements.max()
)

print(
    "Nombre d'événements signalés par plus d'un témoin :",
    nombre_evenements_plusieurs_temoins,
)
print(
    "Nombre de témoins du plus grand événement :",
    plus_grand_evenement,
)

Nombre d'événements signalés par plus d'un témoin : 1102
Nombre de témoins du plus grand événement : 19


## VIII. Afficher le plus gros événement

In [8]:
event_id_plus_grand = taille_evenements.idxmax()

plus_grand_evenement_df = df.loc[
    df["event_id"] == event_id_plus_grand,
    [
        "event_id",
        "datetime",
        "city",
        "state",
        "country",
        "shape",
        "duration_seconds",
        "comments",
        "is_hoax",
    ]
].sort_values("datetime")

plus_grand_evenement_df

,event_id,datetime,city,state,country,shape,duration_seconds,comments,is_hoax
6385,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,circle,0.0,UFO visits in Tinley Park,0
6401,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,unknown,1200.0,3 laser red lights forming a triangle and at t...,0
6400,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,other,1800.0,3 orange lights in the sky over Tinley Park.,0
6399,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,other,1800.0,3 Bright Red Objects in South Eastern Sky that...,0
6398,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,light,900.0,three lights in tinley park,0
6397,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,light,900.0,At about 8pm central time&#44 October 31&#44 2...,0
6396,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,light,900.0,Tinley Pk Red lights - have video,0
6395,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,light,600.0,The triangular formation of red lights witness...,0
6402,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,unknown,600.0,3 red lights,0
6394,2004-10-31 20:00:00 | tinley park | il | us,2004-10-31 20:00:00,tinley park,il,us,light,1800.0,Red Lights (UFOS) above Tinley Park&#44 IL on ...,0


## IV. Compter les témoignages identiques

In [9]:
df["commentaire_normalise"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

taille_commentaires_identiques = (
    df.loc[
        df["commentaire_normalise"].ne(""),
        "commentaire_normalise"
    ]
    .value_counts()
)

commentaires_recopies = taille_commentaires_identiques[
    taille_commentaires_identiques > 1
]

nombre_groupes_recopies = len(commentaires_recopies)

nombre_lignes_recopiees = int(
    commentaires_recopies.sum()
)

nombre_doublons_supplementaires = int(
    (commentaires_recopies - 1).sum()
)

print(
    "Nombre de groupes de commentaires identiques :",
    nombre_groupes_recopies,
)
print(
    "Nombre total de lignes faisant partie d'un groupe recopié :",
    nombre_lignes_recopiees,
)
print(
    "Nombre de doublons au-delà de la première occurrence :",
    nombre_doublons_supplementaires,
)

Nombre de groupes de commentaires identiques : 318
Nombre total de lignes faisant partie d'un groupe recopié : 885
Nombre de doublons au-delà de la première occurrence : 567


## X. Afficher des exemples de témoignages recopiés

In [10]:
commentaire_exemple = commentaires_recopies.index[0]

df.loc[
    df["commentaire_normalise"] == commentaire_exemple,
    [
        "datetime",
        "city",
        "state",
        "country",
        "comments",
        "event_id",
    ]
]

,datetime,city,state,country,comments,event_id
16555,2007-01-15 17:30:00,babbitt,mn,us,lights in the sky,2007-01-15 17:30:00 | babbitt | mn | us
24268,2004-01-23 04:45:00,reamstown,pa,us,Lights in the sky,2004-01-23 04:45:00 | reamstown | pa | us
27429,2011-01-05 19:15:00,corpus christi,tx,us,Lights in the Sky,2011-01-05 19:15:00 | corpus christi | tx | us
31742,2005-02-26 05:40:00,ester,ak,us,lights in the sky,2005-02-26 05:40:00 | ester | ak | us
43454,2013-04-26 22:00:00,lynchburg,va,us,Lights in the sky,2013-04-26 22:00:00 | lynchburg | va | us
45358,2004-04-09 21:00:00,monticello,in,us,Lights in the sky,2004-04-09 21:00:00 | monticello | in | us
46453,2012-05-12 23:20:00,waasis (canada),nb,,Lights in the sky,2012-05-12 23:20:00 | waasis (canada) | nb |
47685,2013-05-18 10:30:00,florence,sc,us,Lights in the sky,2013-05-18 10:30:00 | florence | sc | us
50556,2002-05-05 22:00:00,state college,pa,us,lights in the sky,2002-05-05 22:00:00 | state college | pa | us
51362,2010-05-09 20:30:00,pleasantville,nj,us,Lights in the Sky,2010-05-09 20:30:00 | pleasantville | nj | us


## X.BIS Exportation des doublons

In [11]:
df.loc[
    df["commentaire_normalise"].isin(commentaires_recopies.index),
    [
        "datetime",
        "city",
        "state",
        "country",
        "comments",
        "commentaire_normalise",
        "event_id",
    ]
].sort_values(
    "commentaire_normalise"
).to_csv(
    PHASE7_DIR / "temoignages_recopies_identiques.csv",
    index=False,
)

## XI. Découpe aléatoire précédente

In [12]:
y = df["is_hoax"]

indices_train_aleatoire, indices_test_aleatoire = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

## XII. Compter les événements coupés par la découpe aléatoire

In [13]:
df_split_aleatoire = df[["event_id"]].copy()

df_split_aleatoire["split"] = "train"
df_split_aleatoire.loc[
    indices_test_aleatoire,
    "split"
] = "test"

repartition_evenements_aleatoire = (
    df_split_aleatoire
    .groupby("event_id")["split"]
    .nunique()
)

event_ids_a_cheval = repartition_evenements_aleatoire[
    repartition_evenements_aleatoire > 1
].index

nombre_evenements_a_cheval = len(event_ids_a_cheval)

nombre_releves_a_cheval = int(
    df["event_id"].isin(event_ids_a_cheval).sum()
)

print(
    "Événements répartis entre train et test :",
    nombre_evenements_a_cheval,
)
print(
    "Relevés appartenant à ces événements à cheval :",
    nombre_releves_a_cheval,
)

Événements répartis entre train et test : 373
Relevés appartenant à ces événements à cheval : 889


## XIII. Préparer les variables ML

In [14]:
features_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
]

features_modele = [
    "text_features_without_leakage",
] + features_numeriques

X = df[features_modele].copy()

## XIV. Construire le pipeline

In [15]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features_without_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            features_numeriques,
        ),
    ]
)

def creer_modele():
    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

## XV. Évaluer la découpe aléatoire

In [16]:
modele_aleatoire = creer_modele()

modele_aleatoire.fit(
    X.loc[indices_train_aleatoire],
    y.loc[indices_train_aleatoire],
)

predictions_aleatoires = modele_aleatoire.predict(
    X.loc[indices_test_aleatoire]
)

precision_aleatoire = precision_score(
    y.loc[indices_test_aleatoire],
    predictions_aleatoires,
    zero_division=0,
)

recall_aleatoire = recall_score(
    y.loc[indices_test_aleatoire],
    predictions_aleatoires,
    zero_division=0,
)

print(f"Precision avec découpe aléatoire : {precision_aleatoire:.2%}")
print(f"Recall avec découpe aléatoire : {recall_aleatoire:.2%}")

Precision avec découpe aléatoire : 1.69%
Recall avec découpe aléatoire : 42.53%


## XVI. Découpe par groupes d’événements

In [17]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_positions, test_positions = next(
    group_splitter.split(
        X,
        y,
        groups=df["event_id"],
    )
)

indices_train_groupes = df.index[train_positions]
indices_test_groupes = df.index[test_positions]

print("Train :", len(indices_train_groupes))
print("Test :", len(indices_test_groupes))

Train : 70901
Test : 17778


## XVII. Vérifier qu’aucun événement ne traverse les deux jeux

In [18]:
events_train = set(
    df.loc[indices_train_groupes, "event_id"]
)

events_test = set(
    df.loc[indices_test_groupes, "event_id"]
)

events_partages = events_train.intersection(events_test)

print(
    "Nombre d'événements présents à la fois dans train et test :",
    len(events_partages),
)

assert len(events_partages) == 0

Nombre d'événements présents à la fois dans train et test : 0


## XVIII. Afficher un événement entier avec son côté

In [19]:
df_verification_groupes = df.copy()

df_verification_groupes["jeu"] = "train"
df_verification_groupes.loc[
    indices_test_groupes,
    "jeu"
] = "test"

event_id_exemple = (
    df_verification_groupes["event_id"]
    .value_counts()
    .loc[lambda s: s > 1]
    .index[0]
)

df_verification_groupes.loc[
    df_verification_groupes["event_id"] == event_id_exemple,
    [
        "event_id",
        "jeu",
        "datetime",
        "city",
        "state",
        "country",
        "comments",
    ]
]

,event_id,jeu,datetime,city,state,country,comments
6385,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,UFO visits in Tinley Park
6386,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,Three red lights moving slowly in the sky.
6387,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,3 bright Orange lights over Tinley Park&#44 IL...
6388,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,Red Lights again in Tinley Park&#44 second sig...
6389,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,Slow moving red lights.
6390,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,3red lights sitting still for 20 minutes then ...
6391,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,3red lights sitting still for 20 minutes then ...
6392,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,3 red lights movin E/SE
6393,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,Three red lights in the sky in tinley park. Th...
6394,2004-10-31 20:00:00 | tinley park | il | us,test,2004-10-31 20:00:00,tinley park,il,us,Red Lights (UFOS) above Tinley Park&#44 IL on ...


## XIX. Évaluer le modèle avec la découpe par événements

In [20]:
modele_groupes = creer_modele()

modele_groupes.fit(
    X.loc[indices_train_groupes],
    y.loc[indices_train_groupes],
)

predictions_groupes = modele_groupes.predict(
    X.loc[indices_test_groupes]
)

precision_groupes = precision_score(
    y.loc[indices_test_groupes],
    predictions_groupes,
    zero_division=0,
)

recall_groupes = recall_score(
    y.loc[indices_test_groupes],
    predictions_groupes,
    zero_division=0,
)

accuracy_groupes = accuracy_score(
    y.loc[indices_test_groupes],
    predictions_groupes,
)

print(f"Precision avec groupes : {precision_groupes:.2%}")
print(f"Recall avec groupes : {recall_groupes:.2%}")
print(f"Accuracy avec groupes : {accuracy_groupes:.2%}")

Precision avec groupes : 1.27%
Recall avec groupes : 49.39%
Accuracy avec groupes : 64.16%


## XX. Tableau comparatif final

In [21]:
resultats_phase7 = pd.DataFrame(
    [
        {
            "decoupage": "Aléatoire",
            "precision": precision_aleatoire,
            "recall": recall_aleatoire,
        },
        {
            "decoupage": "Par événements",
            "precision": precision_groupes,
            "recall": recall_groupes,
        },
    ]
)

resultats_phase7

,decoupage,precision,recall
0,Aléatoire,0.016876,0.425287
1,Par événements,0.012718,0.493902


## XXI. Exports

In [22]:
stats_evenements = pd.DataFrame(
    [
        {
            "nombre_evenements_plusieurs_temoins":
                nombre_evenements_plusieurs_temoins,
            "nombre_temoins_plus_gros_evenement":
                plus_grand_evenement,
            "nombre_evenements_a_cheval_decoupage_aleatoire":
                nombre_evenements_a_cheval,
            "nombre_releves_a_cheval_decoupage_aleatoire":
                nombre_releves_a_cheval,
            "nombre_groupes_commentaires_identiques":
                nombre_groupes_recopies,
            "nombre_lignes_commentaires_identiques":
                nombre_lignes_recopiees,
            "nombre_doublons_supplementaires":
                nombre_doublons_supplementaires,
        }
    ]
)

stats_evenements.to_csv(
    PHASE7_DIR / "statistiques_evenements.csv",
    index=False,
)

resultats_phase7.to_csv(
    PHASE7_DIR / "resultats_comparaison_decoupages.csv",
    index=False,
)

plus_grand_evenement_df.to_csv(
    PHASE7_DIR / "plus_grand_evenement.csv",
    index=False,
)